In [1]:
#Transformer -> Transform input into tokens ->Numbers 

In [1]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,Seq2SeqTrainingArguments,Seq2SeqTrainer
from transformers.models.marian import MarianTokenizer
from datasets import load_dataset

In [2]:
#! pip install "datasets<4" "huggingface-hub>=1.5,<2.0" --force-reinstall --no-deps
#! pip install "huggingface-hub>=1.5,<2.0"
#! pip install sentencepiece

#AutoTokenizer -> Tries to auto-detect the right tokenizer for any model

#MarianTokenizer -> Specifically built for Marian/Opus-MT models

#Why it failed here

AutoTokenizer -> In newer transformers versions, MarianConfig was removed from its internal mapping table, so it doesn't know to use MarianTokenizer

MarianTokenizer -> Doesn't rely on any mapping — it is the correct class

In [3]:
#Define the Model Checkpoint

model_checkpoint='Helsinki-NLP/opus-mt-en-mul'
tokenizer=MarianTokenizer.from_pretrained(model_checkpoint) #Changed AutoToknizer to MarianTokenizer 

C:\Users\USER\anaconda3\envs\aiml\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [4]:
#Load the dataset and prints splits 

raw_dataset=load_dataset('Hemanth-thunder/en_ta')
print(raw_dataset)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'en', 'ta'],
        num_rows: 285630
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'en', 'ta'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['Unnamed: 0', 'en', 'ta'],
        num_rows: 2000
    })
})


In [5]:
#Preprocess the data 

max_input_length=128
max_target_length=128
learning_rate=2e-5
batch_size=30
epochs=2
source_lang='en'
target_lang='ta'

In [6]:
#Preprocessing 

def preprocess_function(example):

    inputs=[ex for ex in example[source_lang]]
    target=[ex for ex in example[target_lang]]
    model_inputs=tokenizer(inputs,max_length=max_input_length,truncation=True,padding='max_length')
    #Note : No Need to Use tokenizer as_target_tokenizer() in PyTorch. This was a Tensorflow specific step
    labels=tokenizer(target,max_length=max_target_length,truncation=True,padding='max_length')
    model_inputs['labels']=labels['input_ids']
    return model_inputs

In [7]:
#Process Training Sets 

tokenized_datasets=raw_dataset.map(preprocess_function,batched=True)
tokenized_datasets=tokenized_datasets.remove_columns(['en','ta'])
tokenized_datasets.set_format(type='torch',columns=['input_ids','attention_mask','labels'])

Parameter 'function'=<function preprocess_function at 0x000001959B7E6160> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/285630 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [8]:
#Load the Model (training model)

model=AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [9]:
! pip install "accelerate>=1.1.0"

In [10]:
#Define the training Arugument
#Define the training with fp16 enabled 

training_args=Seq2SeqTrainingArguments(output_dir='/results',
                                       eval_strategy='epoch',
                                      learning_rate=learning_rate,
                                       per_device_train_batch_size=batch_size,
                                      per_device_eval_batch_size=batch_size,
                                       weight_decay=0.01,save_total_limit=3,
                                      num_train_epochs=epochs,
                                       predict_with_generate=True,
                                       logging_steps=500,
                                      fp16=True)#Enabled mixed precision training

In [15]:
#Initialized the Trainer

trainer=Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    processing_class=tokenizer, #used processing_class instaed of tokenizer (cuz they removed it )
)

In [17]:
! pip install --upgrade torch torchvision datasets accelerate transformers

  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 799.2 kB/s eta 0:02:35
   ---------------------------------------- 0.5/124.1 MB 799.2 kB/s eta 0:02:35
   ---------------------------------------- 0.8/124.1 MB 781.4 kB/s eta 0:02:38
   ---------------------------------------- 0.8/124.1 MB 781.4 kB/s eta 0:02:38
   ---------------------------------------- 1.0/124.1 MB 762.8 kB/s eta 0:02:42
   ---------------------------------------- 1.0/124.1 MB 762.8 kB/s eta 0:02:42
   ---------------------------------------- 1.3/124.1 MB 692.1 kB/s eta 0:02:58
   ---------------------------------------- 1.3/124.1 MB 692.1 kB/s eta 0:02:58
    --------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [ ]:
! pip install torch==2.12.1 torchvision==0.27.1 datasets==3.6.0 transformers==5.15.0


In [ ]:
#Train the Model

trainer.train()